In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
from sklearn.neighbors import BallTree
import branca.colormap as cm

In [2]:
df = pd.read_parquet(
    r"../data/processed/v3/housing_residential_processed.parquet"
).copy()

In [ ]:
df_primary = df[df["market_type"] == "primary"]
df_secondary = df[df["market_type"] == "secondary"]

In [13]:
import numpy as np
import folium
from folium.plugins import HeatMap
from branca.colormap import linear


def make_heatmap_by_price(df_primary, df_secondary, column_name: str, save_path: str):
    m = folium.Map(location=[55.75, 37.62], zoom_start=10, tiles=None)
    folium.TileLayer("CartoDB positron", opacity=0.4).add_to(m)

    # значения в млн
    primary_vals = df_primary[column_name].dropna().to_numpy() / 1_000_000
    secondary_vals = df_secondary[column_name].dropna().to_numpy() / 1_000_000
    vals = np.concatenate([primary_vals, secondary_vals])

    vmin, vmax = float(vals.min()), float(vals.max())

    # Легенда: 6 интервалов
    colormap = linear.YlOrRd_09.scale(vmin, vmax).to_step(6)
    colormap.caption = f"{column_name} (M ₽)"
    colormap.add_to(m)

    # Градиент для HeatMap из тех же 6 цветов (чтобы совпадало визуально)
    # Индексы 0..1
    n_steps = 6
    gradient = {}
    for i in range(n_steps):
        x = i / (n_steps - 1)  # 0..1
        # берем цвет ровно по шкале colormap (она уже stepped)
        gradient[x] = colormap(vmin + x * (vmax - vmin))

    def build_heat(df):
        return (
            df[["latitude", "longitude", column_name]]
            .dropna()
            .assign(weight=lambda x: x[column_name] / 1_000_000)[
                ["latitude", "longitude", "weight"]
            ]
            .to_numpy()
            .tolist()
        )

    heat_primary = build_heat(df_primary)
    heat_secondary = build_heat(df_secondary)

    fg1 = folium.FeatureGroup(name="Primary", show=True)
    fg2 = folium.FeatureGroup(name="Secondary", show=False)

    # ВАЖНО: max_val фиксирует нормировку (иначе auto по слою)
    HeatMap(
        heat_primary,
        radius=15,
        min_opacity=0.2,
        gradient=gradient,
    ).add_to(fg1)

    HeatMap(
        heat_secondary,
        radius=15,
        min_opacity=0.2,
        gradient=gradient,
    ).add_to(fg2)

    fg1.add_to(m)
    fg2.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    m.save(save_path)

In [14]:
make_heatmap_by_price(
    df_primary,
    df_secondary,
    column_name="knn_weighted_price",
    save_path="resources/10/primary_vs_secondary_knn_price_heatmaps.html",
)

Теперь посмотрим отношение knn цен первички к вторичке.

In [7]:
def make_heatmap_ratio(
    df_primary,
    ratio_column: str,
    save_path: str,
    radius=15,
    clip_quantiles=(0.02, 0.98),
):
    m = folium.Map(location=[55.75, 37.62], zoom_start=10, tiles=None)

    folium.TileLayer("CartoDB positron", opacity=0.4).add_to(m)

    s = df_primary[["latitude", "longitude", ratio_column]].dropna().copy()
    s = s[np.isfinite(s[ratio_column])]
    s = s[s[ratio_column] > 0]

    # Квантили для обрезки выбросов
    q_low = float(s[ratio_column].quantile(clip_quantiles[0]))
    q_high = float(s[ratio_column].quantile(clip_quantiles[1]))

    # Симметрия вокруг 1
    max_dev = max(abs(1 - q_low), abs(q_high - 1))
    vmin = 1 - max_dev
    vmax = 1 + max_dev

    # Нормализация значений в [0, 1] для HeatMap
    s["w"] = (s[ratio_column] - vmin) / (vmax - vmin)
    s["w"] = s["w"].clip(0, 1)

    heat_data = s[["latitude", "longitude", "w"]].values.tolist()

    # Градиент HeatMap: синий -> белый -> красный
    # ключи — позиции 0..1
    gradient = {
        0.0: "#2c7bb6",  # синий
        0.5: "#ffffbf",  # почти белый/кремовый (лучше виден на карте)
        1.0: "#d7191c",  # красный
    }

    HeatMap(
        heat_data,
        radius=radius,
        gradient=gradient,
        min_opacity=0.2,
        blur=15,
        max_zoom=12,
    ).add_to(m)

    # Легенда (соответствует vmin..vmax и тому же смыслу)
    colormap = cm.LinearColormap(
        colors=[gradient[0.0], gradient[0.5], gradient[1.0]],
        vmin=vmin,
        vmax=vmax,
    )
    colormap.caption = f"{ratio_column} (Primary/Secondary), 1 = parity"
    colormap.add_to(m)

    m.save(save_path)

In [ ]:
make_heatmap_ratio(
    df_primary,
    ratio_column="primary_to_secondary_price_ratio",
    save_path="resources/10/ratio_price_heatmap.html",
)